# Meta Motivo benchmarking using HumEnv

This notebook shows how to evaluate a Meta Motivo model using the benchmark proposed in HumEnv. It assumes that motions for tracking and poses for goal reaching have been processed by following the [instructions](https://github.com/facebookresearch/humenv/tree/main/data_preparation) in HumEnv and are available in the folder `MOTIONS_BASE_PATH`.

In [ ]:
from metamotivo.fb_cpr.huggingface import FBcprModel
from metamotivo.wrappers.humenvbench import RewardWrapper, TrackingWrapper, GoalWrapper
from metamotivo.buffers.buffers import DictBuffer
from huggingface_hub import hf_hub_download
import h5py
import json
import numpy as np
from humenv import STANDARD_TASKS, make_humenv
from gymnasium.wrappers import FlattenObservation
from humenv.bench import (
    RewardEvaluation,
    GoalEvaluation,
    TrackingEvaluation,
)

# paths where to find the output of HumEnv's data preparation scripts
# MOTIONS_BASE_PATH = "humenv/data_preparation/humenv_amass"
# MOTIONS_TRACKING = "humenv/data_preparation/test_train_split/large1_small1_test_0.1.txt"
# GOAL_POSES = "humenv/data_preparation/goal_poses/goals.json"

# # load the goal poses into a dictionary
# with open(GOAL_POSES, "r") as json_file:
#     GOAL_DICT = json.load(json_file)
# GOAL_DICT = {k: np.array(v["observation"]) for k,v in GOAL_DICT.items()}

## Comparing body architectures

This section runs the same HumEnv benchmark against three MJCF rigs that keep
`robot.xml`'s SMPL body/joint/actuator names and gear ratios (so the pretrained
weights, with a fixed obs/action size, still apply) but change bone length and
girth:

- **baseline** -- `robot.xml`, unmodified (what the model was trained on).
- **stocky** -- `robot_stocky.xml`, legs/arms shortened (x0.75 / x0.85), girth
  increased (x1.35). Generated with `scale_robot.py`.
- **tall** -- `robot_tall.xml`, legs/arms/torso lengthened (x1.4 / x1.2 / x1.1),
  girth reduced (x0.85).

Both variants were produced by `scale_robot.py --input robot.xml --output ...`
in this folder (re-run it with different `--*_scale` flags to try other body
shapes). A compatibility preflight (same idea as `motivo_xml_swap.ipynb`) checks
that each variant keeps the exact observation/action dimensions the pretrained
model expects before any evaluation runs.


In [2]:
# One entry per rig to benchmark. `None` means "use HumEnv's bundled default
# rig", which is the exact skeleton robot.xml mirrors -- keep it first as the
# baseline everything else is compared against.
XML_VARIANTS = {
    "baseline (robot.xml)": "../assets/robots/robot.xml",
    "stocky (robot_stocky.xml)": "../assets/robots/robot_stocky.xml",
    "tall (robot_tall.xml)": "../assets/robots/robot_tall.xml",
}

# Full STANDARD_TASKS x num_episodes x 3 rigs is expensive. Set this to False
# once the quick pass below looks sane and you want the paper-scale numbers.
QUICK_COMPARISON = True
REWARD_TASKS = STANDARD_TASKS[:3] if QUICK_COMPARISON else STANDARD_TASKS
NUM_EPISODES = 5 if QUICK_COMPARISON else 100
NUM_ENVS = 5 if QUICK_COMPARISON else 50


In [3]:
def check_xml_compatibility(xml_path):
    """Raise if xml_path's obs/action dims don't match HumEnv's default rig
    (what the pretrained model's fixed-size networks expect). Mirrors the
    preflight in motivo_xml_swap.ipynb."""
    if xml_path is None:
        return
    ref_env, _ = make_humenv(num_envs=1, wrappers=[FlattenObservation])
    expected_obs = ref_env.observation_space.shape[0]
    expected_actions = ref_env.action_space.shape[0]
    ref_env.close()

    env, _ = make_humenv(num_envs=1, xml=xml_path, wrappers=[FlattenObservation])
    actual_obs = env.observation_space.shape[0]
    actual_actions = env.action_space.shape[0]
    env.close()

    print(f"{xml_path}: obs {actual_obs} (expected {expected_obs}), "
          f"actions {actual_actions} (expected {expected_actions})")
    if (actual_obs, actual_actions) != (expected_obs, expected_actions):
        raise ValueError(
            f"{xml_path} does not match the pretrained model's fixed obs/action "
            "size -- it must keep robot.xml's body/joint/actuator names and gear "
            "ratios, only varying morphology."
        )

for label, xml_path in XML_VARIANTS.items():
    check_xml_compatibility(xml_path)


robot.xml: obs 358 (expected 358), actions 69 (expected 69)
robot_stocky.xml: obs 358 (expected 358), actions 69 (expected 69)
robot_tall.xml: obs 358 (expected 358), actions 69 (expected 69)


Load inference buffer.

In [5]:
buffer_path = hf_hub_download(
        repo_id="facebook/metamotivo-M-1",
        filename="data/buffer_inference_500000.hdf5",
        repo_type="model",
        local_dir="../data/metamotivo-M-1-datasets",
    )
hf = h5py.File(buffer_path, "r")
data = {k: v[:] for k, v in hf.items()}
buffer = DictBuffer(capacity=data["qpos"].shape[0], device="cpu")
buffer.extend(data)

Load model and prepare it for inference.

In [6]:
device = "cuda"  # it is normally faster to evaluate on cpu since tracking is parallelized
model = FBcprModel.from_pretrained("facebook/metamotivo-M-1").to(device)
model = RewardWrapper(
        model=model,
        inference_dataset=buffer,
        num_samples_per_inference=100_000,
        inference_function="reward_wr_inference",
        max_workers=10,
    )
# model = GoalWrapper(model=model)
# model = TrackingWrapper(model=model)

Humenv provides 3 evaluation protocols:
- reward based,
- goal based,
- tracking

## Run the benchmark once per rig

Same three HumEnv protocols as before (reward / goal / tracking), just run
once per entry in `XML_VARIANTS` so the results are directly comparable. Model
and inference buffer are unaffected by the rig -- only the environments built
inside each `*Evaluation` need the swapped `xml`.


In [ ]:
results = {}

for label, xml_path in XML_VARIANTS.items():
    print(f"\n=== {label} ===")
    env_kwargs = {"state_init": "Fall"}
    if xml_path is not None:
        env_kwargs["xml"] = xml_path

    reward_eval = RewardEvaluation(
        tasks=REWARD_TASKS,
        env_kwargs=env_kwargs,
        num_contexts=1,
        num_envs=NUM_ENVS,
        num_episodes=NUM_EPISODES,
    )
    reward_metrics = reward_eval.run(agent=model)
    reward_mean = np.array([m["reward"] for m in reward_metrics.values()]).mean()

    # goal_eval = GoalEvaluation(
    #     goals=GOAL_DICT,
    #     env_kwargs={**env_kwargs, "state_init": "Fall"},
    #     num_contexts=1,
    #     num_envs=NUM_ENVS,
    #     num_episodes=NUM_EPISODES,
    # )
    # goal_metrics = goal_eval.run(agent=model)
    # goal_success = np.array([m["success"] for m in goal_metrics.values()]).mean()
    # goal_proximity = np.array([m["proximity"] for m in goal_metrics.values()]).mean()

    # tracking_eval = TrackingEvaluation(
    #     motions=MOTIONS_TRACKING,
    #     motion_base_path=MOTIONS_BASE_PATH,
    #     env_kwargs={**env_kwargs, "state_init": "Default"},
    #     num_envs=NUM_ENVS,
    # )
    # tracking_metrics = tracking_eval.run(agent=model)
    # tracking_success = np.array([m["success_phc_linf"] for m in tracking_metrics.values()]).mean()
    # tracking_emd = np.array([m["emd"] for m in tracking_metrics.values()]).mean()

    results[label] = {
        "reward_mean": reward_mean,
        # "goal_success": goal_success,
        # "goal_proximity": goal_proximity,
        # "tracking_success_phc_linf": tracking_success,
        # "tracking_emd": tracking_emd,
    }
    print(results[label])


In [24]:
baseline_label = next(iter(XML_VARIANTS))
baseline = results[baseline_label]

print(f"{'variant':28s} " + " ".join(f"{k:>24s}" for k in baseline))
for label, m in results.items():
    row = f"{label:28s} "
    for k, v in m.items():
        delta = v - baseline[k]
        pct = (delta / baseline[k] * 100) if baseline[k] else float("nan")
        row += f"{v:10.4f} ({pct:+6.1f}%) "
    print(row)


variant                                   reward_mean
baseline (robot.xml)           192.6464 (  +0.0%) 
stocky (robot_stocky.xml)       94.2487 ( -51.1%) 
tall (robot_tall.xml)          105.3175 ( -45.3%) 
